# MedicalNLP-RAG — Phase 4E
## Task-Specific Selective Answering / Abstention

Phase 4D v2 showed that confidence utility is **task-dependent**:

- **summary:** clear within-task discrimination;
- **factoid:** weaker but statistically supported discrimination;
- **yes/no:** no useful instance-level discrimination beyond the task mean;
- **list:** no useful instance-level discrimination beyond the task mean.

Phase 4E therefore does **not** force one global abstention rule across heterogeneous BioASQ tasks.

The 860-question internal test remains locked.

---

## Frozen upstream system

No retrieval, generation, grounding, or confidence model is re-trained here.

`BM25 + MedCPT → RRF → MedCPT Cross-Encoder → document_top5 → Qwen3-4B-Instruct → Phase 4C v2 grounding → Phase 4D v2 task-specific confidence`

Phase 4E evaluates only the **out-of-fold validation predictions** generated in Phase 4D v2.

---

# Core selective-answering definitions

For a question with benchmark task score `s ∈ [0,1]`:

`risk = 1 - s`

At coverage `c`, only the highest-ranked fraction `c` of questions is answered.

For each task we report:

- coverage;
- abstention rate;
- selective mean task score;
- selective risk;
- gain over full coverage;
- relative risk reduction;
- risk–coverage curve;
- discrete AURC;
- oracle AURC;
- random-ranking AURC;
- normalized AURC.

Lower AURC is better.

---

# Ranking versus displayed confidence

Phase 4D v2 froze two outputs:

- **ranking signal:** task-specific raw Ridge score;
- **reported confidence:** task-specific isotonic expected task score.

Selective ranking uses the raw Ridge score because isotonic regression can create plateaus.

The calibrated isotonic value remains the displayed expected task score.

---

# Prespecified operating point

Before inspecting Phase 4E risk–coverage results:

**Primary task coverage = 80%**

This corresponds to 20% abstention **within a task**.

Sensitivity operating points:

`90%, 80%, 70%, 60%, 50%`

The best coverage is **not** selected post hoc.

---

# Eligibility rule inherited from Phase 4D v2

A task is considered:

### PRIMARY-ELIGIBLE
if:
1. within-task raw-Ridge Spearman is positive and Holm-adjusted `p < 0.05`; and
2. raw Ridge significantly improves MSE over the train-mean baseline by paired bootstrap.

### SECONDARY-ELIGIBLE
if:
1. within-task raw-Ridge Spearman is positive and Holm-adjusted `p < 0.05`; and
2. raw Ridge significantly improves MAE, but not MSE, over the train-mean baseline.

### INELIGIBLE
otherwise.

This rule is evaluated from the already-completed Phase 4D v2 results.

Expected from the frozen Phase 4D v2 analysis:

- summary → PRIMARY
- factoid → SECONDARY
- yes/no → INELIGIBLE
- list → INELIGIBLE

The notebook verifies rather than assumes this classification.

---

# Two frozen system-level policies

## Policy A — Conservative / primary
- summary: answer top 80% by task-specific raw Ridge score;
- factoid: answer all;
- yes/no: answer all;
- list: answer all.

## Policy B — Expanded / secondary
- summary: answer top 80%;
- factoid: answer top 80%;
- yes/no: answer all;
- list: answer all.

Policy B is secondary because factoid confidence showed weaker evidence than summary confidence.

No abstention is forced for yes/no or list.

## 0. Configuration

In [ ]:
PROJECT_NAME = "MedicalNLP_RAG"
SEED = 42

PRIMARY_TASK_COVERAGE = 0.80

SENSITIVITY_COVERAGES = [
    0.90,
    0.80,
    0.70,
    0.60,
    0.50,
]

TASKS = [
    "yesno",
    "factoid",
    "list",
    "summary",
]

RANKING_SCORE = "task_ridge_raw"
CALIBRATED_CONFIDENCE = "task_ridge_isotonic"

RANDOM_PERMUTATION_REPS = 2000
BOOTSTRAP_REPS = 5000

ALPHA = 0.05

ALLOW_INTERNAL_TEST_EVALUATION = False

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 2. Imports

In [ ]:
from pathlib import Path
import json
import math
import random
import warnings
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import spearmanr

warnings.filterwarnings("ignore")

random.seed(SEED)
np.random.seed(SEED)

## 3. Paths and required frozen inputs

In [ ]:
ROOT = Path(
    f"/content/drive/MyDrive/{PROJECT_NAME}"
)

PHASE4D_DIR = (
    ROOT / "results"
    / "phase4D_v2_task_specific_confidence"
)

PHASE4D_TABLE_DIR = (
    PHASE4D_DIR / "tables"
)

PHASE4E_DIR = (
    ROOT / "results"
    / "phase4E_task_specific_selective_answering"
)

TABLE_DIR = PHASE4E_DIR / "tables"
FIG_DIR = PHASE4E_DIR / "figures"
LOG_DIR = PHASE4E_DIR / "logs"

for p in [
    PHASE4E_DIR,
    TABLE_DIR,
    FIG_DIR,
    LOG_DIR,
]:
    p.mkdir(
        parents=True,
        exist_ok=True
    )

OOF_PREDICTIONS = (
    PHASE4D_TABLE_DIR
    / "task_specific_oof_predictions.csv"
)

TASK_METRICS = (
    PHASE4D_TABLE_DIR
    / "task_specific_confidence_metrics.csv"
)

BOOTSTRAP_ERRORS = (
    PHASE4D_TABLE_DIR
    / "task_specific_paired_bootstrap_errors.csv"
)

PHASE4D_MODELING_TABLE = (
    PHASE4D_TABLE_DIR
    / "phase4D_v2_modeling_table.csv"
)

FROZEN_CONFIDENCE = (
    PHASE4D_DIR
    / "frozen_confidence_method_v2.json"
)

PHASE4D_REPORT = (
    PHASE4D_DIR
    / "phase4D_v2_report.json"
)

for required in [
    OOF_PREDICTIONS,
    TASK_METRICS,
    BOOTSTRAP_ERRORS,
    PHASE4D_MODELING_TABLE,
    FROZEN_CONFIDENCE,
    PHASE4D_REPORT,
]:
    assert required.exists(), (
        f"Required Phase 4D v2 artifact is missing: {required}\n"
        "Run Phase 4D v2 through its freeze/report cells before Phase 4E."
    )

print(
    "Phase 4E output:",
    PHASE4E_DIR
)

## 4. Locked-test safety

In [ ]:
if ALLOW_INTERNAL_TEST_EVALUATION:
    raise RuntimeError(
        "Internal test must remain locked during Phase 4E."
    )

print("INTERNAL TEST REMAINS LOCKED.")

## 5. Verify frozen Phase 4D v2

In [ ]:
with open(
    FROZEN_CONFIDENCE,
    "r",
    encoding="utf-8"
) as f:
    frozen_confidence = json.load(f)

assert (
    frozen_confidence[
        "internal_test_evaluated"
    ]
    is False
)

assert (
    frozen_confidence[
        "question_type_used_as_feature"
    ]
    is False
)

assert (
    frozen_confidence[
        "abstention_threshold_selected"
    ]
    is False
)

print(
    "Frozen Phase 4D v2 verified."
)

print(
    "Ranking signal:",
    frozen_confidence[
        "frozen_operational_outputs"
    ][
        "ranking_signal"
    ]
)

print(
    "Reported confidence:",
    frozen_confidence[
        "frozen_operational_outputs"
    ][
        "reported_confidence"
    ]
)

## 6. Load Phase 4D v2 OOF predictions

In [ ]:
oof_df = pd.read_csv(
    OOF_PREDICTIONS,
    dtype={"id": str}
)

assert len(oof_df) == 862

required_cols = [
    "id",
    "type",
    "task_score",
    "outer_fold",
    "train_mean_baseline",
    "mean_top1_entailment",
    "top1_supported_claim_rate",
    RANKING_SCORE,
    CALIBRATED_CONFIDENCE,
]

missing_cols = [
    c
    for c in required_cols
    if c not in oof_df.columns
]

assert not missing_cols, (
    missing_cols
)

assert oof_df[
    "task_score"
].between(
    0,
    1
).all()

assert oof_df[
    CALIBRATED_CONFIDENCE
].between(
    0,
    1
).all()

display(
    oof_df[
        "type"
    ].value_counts()
)

## 7. Load Phase 4D v2 metrics and bootstrap results

In [ ]:
task_metric_df = pd.read_csv(
    TASK_METRICS
)

bootstrap_df = pd.read_csv(
    BOOTSTRAP_ERRORS
)

display(
    task_metric_df[
        [
            "task",
            "method",
            "n",
            "rmse",
            "ece_10bin",
            "spearman_rho",
            "spearman_p",
        ]
    ]
)

display(
    bootstrap_df[
        [
            "task",
            "model_a",
            "model_b",
            "mae_mean_delta_a_minus_b",
            "mae_ci95_low",
            "mae_ci95_high",
            "mse_mean_delta_a_minus_b",
            "mse_ci95_low",
            "mse_ci95_high",
        ]
    ]
)

# Part A — Freeze task eligibility before risk–coverage analysis

## 8. Holm correction for the four within-task Spearman tests

In [ ]:
raw_metric = (
    task_metric_df[
        task_metric_df[
            "method"
        ] == RANKING_SCORE
    ][
        [
            "task",
            "spearman_rho",
            "spearman_p",
            "mae",
            "rmse",
            "ece_10bin",
        ]
    ]
    .copy()
)


def holm_adjust(
    p_values
):
    p_values = np.asarray(
        p_values,
        dtype=float
    )

    m = len(
        p_values
    )

    order = np.argsort(
        p_values
    )

    adjusted_sorted = np.empty(
        m,
        dtype=float
    )

    running_max = 0.0

    for rank, idx in enumerate(
        order
    ):
        adjusted = (
            (m - rank)
            * p_values[
                idx
            ]
        )

        running_max = max(
            running_max,
            adjusted
        )

        adjusted_sorted[
            idx
        ] = min(
            1.0,
            running_max
        )

    return adjusted_sorted


raw_metric[
    "spearman_p_holm"
] = holm_adjust(
    raw_metric[
        "spearman_p"
    ].to_numpy(float)
)

display(
    raw_metric.sort_values(
        "task"
    )
)

## 9. Apply prespecified eligibility rule

In [ ]:
raw_vs_mean = (
    bootstrap_df[
        (
            bootstrap_df[
                "model_a"
            ] == RANKING_SCORE
        )
        &
        (
            bootstrap_df[
                "model_b"
            ] == "train_mean_baseline"
        )
    ][
        [
            "task",
            "mae_mean_delta_a_minus_b",
            "mae_ci95_low",
            "mae_ci95_high",
            "mse_mean_delta_a_minus_b",
            "mse_ci95_low",
            "mse_ci95_high",
        ]
    ]
    .copy()
)

eligibility_df = (
    raw_metric.merge(
        raw_vs_mean,
        on="task",
        validate="one_to_one",
    )
)

def classify_task(row):
    discrimination_ok = (
        row[
            "spearman_rho"
        ] > 0
        and
        row[
            "spearman_p_holm"
        ] < ALPHA
    )

    mse_better = (
        row[
            "mse_ci95_high"
        ] < 0
    )

    mae_better = (
        row[
            "mae_ci95_high"
        ] < 0
    )

    if (
        discrimination_ok
        and mse_better
    ):
        return "PRIMARY_ELIGIBLE"

    if (
        discrimination_ok
        and mae_better
    ):
        return "SECONDARY_ELIGIBLE"

    return "INELIGIBLE"


eligibility_df[
    "eligibility"
] = eligibility_df.apply(
    classify_task,
    axis=1
)

eligibility_df.to_csv(
    TABLE_DIR
    / "task_selective_eligibility.csv",
    index=False
)

display(
    eligibility_df[
        [
            "task",
            "spearman_rho",
            "spearman_p",
            "spearman_p_holm",
            "mae_ci95_low",
            "mae_ci95_high",
            "mse_ci95_low",
            "mse_ci95_high",
            "eligibility",
        ]
    ]
)

## 10. Safety assertion for frozen interpretation

In [ ]:
eligibility_map = (
    eligibility_df
    .set_index(
        "task"
    )[
        "eligibility"
    ]
    .to_dict()
)

print(
    json.dumps(
        eligibility_map,
        indent=2
    )
)

assert (
    eligibility_map[
        "summary"
    ]
    == "PRIMARY_ELIGIBLE"
), eligibility_map

assert (
    eligibility_map[
        "factoid"
    ]
    == "SECONDARY_ELIGIBLE"
), eligibility_map

assert (
    eligibility_map[
        "yesno"
    ]
    == "INELIGIBLE"
), eligibility_map

assert (
    eligibility_map[
        "list"
    ]
    == "INELIGIBLE"
), eligibility_map

print(
    "Eligibility pattern matches the frozen Phase 4D v2 interpretation."
)

# Part B — Risk–coverage functions

## 11. Selective curve helper

In [ ]:
def selective_curve(
    y_true,
    ranking_score,
):
    y_true = np.asarray(
        y_true,
        dtype=float
    )

    ranking_score = np.asarray(
        ranking_score,
        dtype=float
    )

    order = np.argsort(
        -ranking_score,
        kind="mergesort",
    )

    y_sorted = y_true[
        order
    ]

    score_sorted = ranking_score[
        order
    ]

    n = len(
        y_sorted
    )

    k = np.arange(
        1,
        n + 1
    )

    coverage = (
        k / n
    )

    selective_score = (
        np.cumsum(
            y_sorted
        )
        / k
    )

    selective_risk = (
        1.0
        - selective_score
    )

    return pd.DataFrame({
        "rank": k,
        "coverage": coverage,
        "abstention_rate": (
            1.0
            - coverage
        ),
        "selective_task_score": (
            selective_score
        ),
        "selective_risk": (
            selective_risk
        ),
        "ranking_score_cutoff": (
            score_sorted
        ),
    })

## 12. Discrete AURC helper

In [ ]:
def discrete_aurc(
    y_true,
    ranking_score,
):
    curve = selective_curve(
        y_true,
        ranking_score
    )

    # Mean selective risk over all attainable prefix coverages.
    return float(
        curve[
            "selective_risk"
        ].mean()
    )

## 13. Oracle and random-ranking reference

In [ ]:
def random_aurc_distribution(
    y_true,
    n_reps=RANDOM_PERMUTATION_REPS,
    seed=SEED,
):
    y_true = np.asarray(
        y_true,
        dtype=float
    )

    n = len(
        y_true
    )

    rng = np.random.default_rng(
        seed
    )

    values = np.empty(
        n_reps,
        dtype=float
    )

    for i in range(
        n_reps
    ):
        random_score = (
            rng.random(
                n
            )
        )

        values[
            i
        ] = discrete_aurc(
            y_true,
            random_score,
        )

    return values


def aurc_summary(
    y_true,
    ranking_score,
    seed,
):
    y_true = np.asarray(
        y_true,
        dtype=float
    )

    ranking_score = np.asarray(
        ranking_score,
        dtype=float
    )

    observed = discrete_aurc(
        y_true,
        ranking_score,
    )

    # Highest true task score first = oracle ordering.
    oracle = discrete_aurc(
        y_true,
        y_true,
    )

    random_values = (
        random_aurc_distribution(
            y_true,
            seed=seed,
        )
    )

    random_mean = float(
        random_values.mean()
    )

    denom = (
        random_mean
        - oracle
    )

    if abs(
        denom
    ) < 1e-12:
        normalized = np.nan
    else:
        normalized = float(
            (
                observed
                - oracle
            )
            / denom
        )

    # Lower AURC is better.
    p_better_than_random = float(
        (
            1
            + np.sum(
                random_values
                <= observed
            )
        )
        / (
            len(
                random_values
            )
            + 1
        )
    )

    return {
        "aurc": observed,
        "oracle_aurc": oracle,
        "random_aurc_mean": (
            random_mean
        ),
        "random_aurc_ci95_low": float(
            np.quantile(
                random_values,
                0.025
            )
        ),
        "random_aurc_ci95_high": float(
            np.quantile(
                random_values,
                0.975
            )
        ),
        "normalized_aurc": (
            normalized
        ),
        "permutation_p_better_than_random": (
            p_better_than_random
        ),
    }

# Part C — Full risk–coverage analysis by task

Methods compared:

- task-specific Ridge quality score;
- mean Top-1 entailment;
- Top-1 supported-claim rate;
- random ordering;
- oracle ordering.

The oracle uses true validation task scores and is shown only as an upper bound.

## 14. Build and save task-specific risk–coverage curves

In [ ]:
curve_parts = []

for task in TASKS:
    subset = (
        oof_df[
            oof_df[
                "type"
            ] == task
        ]
        .copy()
    )

    for method in [
        RANKING_SCORE,
        "mean_top1_entailment",
        "top1_supported_claim_rate",
    ]:
        curve = selective_curve(
            subset[
                "task_score"
            ],
            subset[
                method
            ],
        )

        curve[
            "task"
        ] = task

        curve[
            "method"
        ] = method

        curve_parts.append(
            curve
        )

    oracle_curve = selective_curve(
        subset[
            "task_score"
        ],
        subset[
            "task_score"
        ],
    )

    oracle_curve[
        "task"
    ] = task

    oracle_curve[
        "method"
    ] = "oracle"

    curve_parts.append(
        oracle_curve
    )

risk_coverage_df = pd.concat(
    curve_parts,
    ignore_index=True
)

risk_coverage_df.to_csv(
    TABLE_DIR
    / "risk_coverage_curves.csv",
    index=False
)

display(
    risk_coverage_df.head()
)

## 15. AURC and random-ranking comparisons

In [ ]:
aurc_rows = []

methods_for_aurc = [
    RANKING_SCORE,
    "mean_top1_entailment",
    "top1_supported_claim_rate",
]

for task_index, task in enumerate(
    TASKS
):
    subset = (
        oof_df[
            oof_df[
                "type"
            ] == task
        ]
    )

    for method_index, method in enumerate(
        methods_for_aurc
    ):
        stats = aurc_summary(
            subset[
                "task_score"
            ],
            subset[
                method
            ],
            seed=(
                SEED
                + 100 * task_index
                + method_index
            ),
        )

        stats.update({
            "task": task,
            "method": method,
            "n": int(
                len(subset)
            ),
        })

        aurc_rows.append(
            stats
        )

aurc_df = (
    pd.DataFrame(
        aurc_rows
    )
    .sort_values(
        [
            "task",
            "aurc",
        ]
    )
)

aurc_df.to_csv(
    TABLE_DIR
    / "aurc_by_task.csv",
    index=False
)

display(
    aurc_df
)

## 16. Risk–coverage figure — summary

In [ ]:
task = "summary"

fig, ax = plt.subplots(
    figsize=(7.0, 5.5)
)

for method in [
    RANKING_SCORE,
    "mean_top1_entailment",
    "top1_supported_claim_rate",
    "oracle",
]:
    part = (
        risk_coverage_df[
            (
                risk_coverage_df[
                    "task"
                ] == task
            )
            &
            (
                risk_coverage_df[
                    "method"
                ] == method
            )
        ]
    )

    ax.plot(
        part[
            "coverage"
        ],
        part[
            "selective_risk"
        ],
        label=method,
    )

full_risk = float(
    1.0
    - oof_df.loc[
        oof_df[
            "type"
        ] == task,
        "task_score"
    ].mean()
)

ax.axhline(
    full_risk,
    linestyle="--",
    label="full-coverage risk",
)

ax.set_xlim(
    0.1,
    1.0
)

ax.set_ylim(
    0,
    1
)

ax.set_xlabel(
    "Coverage"
)

ax.set_ylabel(
    "Selective risk (1 - mean ROUGE-L)"
)

ax.set_title(
    "Summary risk–coverage"
)

ax.legend()

fig.tight_layout()

fig.savefig(
    FIG_DIR
    / "fig01_summary_risk_coverage.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

## 17. Risk–coverage figure — factoid

In [ ]:
task = "factoid"

fig, ax = plt.subplots(
    figsize=(7.0, 5.5)
)

for method in [
    RANKING_SCORE,
    "mean_top1_entailment",
    "top1_supported_claim_rate",
    "oracle",
]:
    part = (
        risk_coverage_df[
            (
                risk_coverage_df[
                    "task"
                ] == task
            )
            &
            (
                risk_coverage_df[
                    "method"
                ] == method
            )
        ]
    )

    ax.plot(
        part[
            "coverage"
        ],
        part[
            "selective_risk"
        ],
        label=method,
    )

full_risk = float(
    1.0
    - oof_df.loc[
        oof_df[
            "type"
        ] == task,
        "task_score"
    ].mean()
)

ax.axhline(
    full_risk,
    linestyle="--",
    label="full-coverage risk",
)

ax.set_xlim(
    0.1,
    1.0
)

ax.set_ylim(
    0,
    1
)

ax.set_xlabel(
    "Coverage"
)

ax.set_ylabel(
    "Selective risk (1 - mean MRR)"
)

ax.set_title(
    "Factoid risk–coverage"
)

ax.legend()

fig.tight_layout()

fig.savefig(
    FIG_DIR
    / "fig02_factoid_risk_coverage.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

## 18. Negative-control risk–coverage figure — yes/no

In [ ]:
task = "yesno"

fig, ax = plt.subplots(
    figsize=(7.0, 5.5)
)

for method in [
    RANKING_SCORE,
    "mean_top1_entailment",
    "top1_supported_claim_rate",
    "oracle",
]:
    part = (
        risk_coverage_df[
            (
                risk_coverage_df[
                    "task"
                ] == task
            )
            &
            (
                risk_coverage_df[
                    "method"
                ] == method
            )
        ]
    )

    ax.plot(
        part[
            "coverage"
        ],
        part[
            "selective_risk"
        ],
        label=method,
    )

full_risk = float(
    1.0
    - oof_df.loc[
        oof_df[
            "type"
        ] == task,
        "task_score"
    ].mean()
)

ax.axhline(
    full_risk,
    linestyle="--",
    label="full-coverage risk",
)

ax.set_xlim(
    0.1,
    1.0
)

ax.set_ylim(
    0,
    1
)

ax.set_xlabel(
    "Coverage"
)

ax.set_ylabel(
    "Selective error rate"
)

ax.set_title(
    "Yes/no risk–coverage — negative control"
)

ax.legend()

fig.tight_layout()

fig.savefig(
    FIG_DIR
    / "fig03_yesno_risk_coverage_negative_control.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

## 19. Negative-control risk–coverage figure — list

In [ ]:
task = "list"

fig, ax = plt.subplots(
    figsize=(7.0, 5.5)
)

for method in [
    RANKING_SCORE,
    "mean_top1_entailment",
    "top1_supported_claim_rate",
    "oracle",
]:
    part = (
        risk_coverage_df[
            (
                risk_coverage_df[
                    "task"
                ] == task
            )
            &
            (
                risk_coverage_df[
                    "method"
                ] == method
            )
        ]
    )

    ax.plot(
        part[
            "coverage"
        ],
        part[
            "selective_risk"
        ],
        label=method,
    )

full_risk = float(
    1.0
    - oof_df.loc[
        oof_df[
            "type"
        ] == task,
        "task_score"
    ].mean()
)

ax.axhline(
    full_risk,
    linestyle="--",
    label="full-coverage risk",
)

ax.set_xlim(
    0.1,
    1.0
)

ax.set_ylim(
    0,
    1
)

ax.set_xlabel(
    "Coverage"
)

ax.set_ylabel(
    "Selective risk (1 - mean F1)"
)

ax.set_title(
    "List risk–coverage — negative control"
)

ax.legend()

fig.tight_layout()

fig.savefig(
    FIG_DIR
    / "fig04_list_risk_coverage_negative_control.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# Part D — Prespecified coverage operating points

No task-specific coverage is optimized.

We evaluate the already-fixed:

`90%, 80%, 70%, 60%, 50%`

## 20. Fixed-coverage selector

In [ ]:
def select_top_fraction(
    frame,
    coverage,
    score_col=RANKING_SCORE,
):
    assert (
        0 < coverage <= 1
    )

    frame = (
        frame.copy()
        .sort_values(
            score_col,
            ascending=False,
            kind="mergesort",
        )
        .reset_index(drop=True)
    )

    n = len(
        frame
    )

    k = int(
        math.ceil(
            coverage
            * n
        )
    )

    selected = (
        frame.iloc[
            :k
        ]
        .copy()
    )

    rejected = (
        frame.iloc[
            k:
        ]
        .copy()
    )

    return (
        selected,
        rejected,
    )

## 21. Evaluate fixed coverage levels

In [ ]:
coverage_rows = []

for task in TASKS:
    subset = (
        oof_df[
            oof_df[
                "type"
            ] == task
        ]
        .copy()
    )

    full_score = float(
        subset[
            "task_score"
        ].mean()
    )

    full_risk = float(
        1.0
        - full_score
    )

    for target_coverage in (
        SENSITIVITY_COVERAGES
    ):
        selected, rejected = (
            select_top_fraction(
                subset,
                target_coverage,
            )
        )

        selected_score = float(
            selected[
                "task_score"
            ].mean()
        )

        selected_risk = float(
            1.0
            - selected_score
        )

        risk_reduction = (
            full_risk
            - selected_risk
        )

        if full_risk > 0:
            relative_rr = float(
                risk_reduction
                / full_risk
            )
        else:
            relative_rr = np.nan

        coverage_rows.append({
            "task": task,
            "target_coverage": (
                target_coverage
            ),
            "n_total": int(
                len(subset)
            ),
            "n_answered": int(
                len(selected)
            ),
            "n_abstained": int(
                len(rejected)
            ),
            "realized_coverage": float(
                len(selected)
                / len(subset)
            ),
            "abstention_rate": float(
                len(rejected)
                / len(subset)
            ),
            "full_task_score": (
                full_score
            ),
            "selective_task_score": (
                selected_score
            ),
            "task_score_gain": float(
                selected_score
                - full_score
            ),
            "full_risk": (
                full_risk
            ),
            "selective_risk": (
                selected_risk
            ),
            "absolute_risk_reduction": float(
                risk_reduction
            ),
            "relative_risk_reduction": (
                relative_rr
            ),
            "minimum_selected_raw_score": float(
                selected[
                    RANKING_SCORE
                ].min()
            ),
            "minimum_selected_calibrated_confidence": float(
                selected[
                    CALIBRATED_CONFIDENCE
                ].min()
            ),
        })

coverage_table = pd.DataFrame(
    coverage_rows
)

coverage_table.to_csv(
    TABLE_DIR
    / "fixed_coverage_task_results.csv",
    index=False
)

display(
    coverage_table
)

# Part E — Fixed 80% coverage statistical analysis

This is the prespecified primary task operating point.

We test whether the score gain produced by the fixed ranking is better than random selection at the same coverage.

No coverage tuning is performed.

## 22. Random-subset test helper

In [ ]:
def random_subset_test(
    y_true,
    ranking_score,
    coverage=PRIMARY_TASK_COVERAGE,
    n_reps=RANDOM_PERMUTATION_REPS,
    seed=SEED,
):
    y_true = np.asarray(
        y_true,
        dtype=float
    )

    ranking_score = np.asarray(
        ranking_score,
        dtype=float
    )

    n = len(
        y_true
    )

    k = int(
        math.ceil(
            coverage
            * n
        )
    )

    order = np.argsort(
        -ranking_score,
        kind="mergesort",
    )

    observed = float(
        y_true[
            order[
                :k
            ]
        ].mean()
    )

    rng = np.random.default_rng(
        seed
    )

    random_means = np.empty(
        n_reps,
        dtype=float
    )

    for i in range(
        n_reps
    ):
        idx = rng.choice(
            n,
            size=k,
            replace=False,
        )

        random_means[
            i
        ] = float(
            y_true[
                idx
            ].mean()
        )

    p_random_at_least_as_good = float(
        (
            1
            + np.sum(
                random_means
                >= observed
            )
        )
        / (
            n_reps
            + 1
        )
    )

    return {
        "n": n,
        "n_answered": k,
        "realized_coverage": float(
            k / n
        ),
        "observed_selective_score": (
            observed
        ),
        "full_score": float(
            y_true.mean()
        ),
        "gain_over_full": float(
            observed
            - y_true.mean()
        ),
        "random_subset_mean": float(
            random_means.mean()
        ),
        "random_subset_ci95_low": float(
            np.quantile(
                random_means,
                0.025
            )
        ),
        "random_subset_ci95_high": float(
            np.quantile(
                random_means,
                0.975
            )
        ),
        "permutation_p_random_at_least_as_good": (
            p_random_at_least_as_good
        ),
    }

## 23. 80% coverage tests for all four tasks

In [ ]:
coverage80_rows = []

for task_index, task in enumerate(
    TASKS
):
    subset = (
        oof_df[
            oof_df[
                "type"
            ] == task
        ]
    )

    stats = random_subset_test(
        subset[
            "task_score"
        ],
        subset[
            RANKING_SCORE
        ],
        coverage=(
            PRIMARY_TASK_COVERAGE
        ),
        seed=(
            SEED
            + 100 * task_index
        ),
    )

    stats.update({
        "task": task,
        "eligibility": (
            eligibility_map[
                task
            ]
        ),
    })

    coverage80_rows.append(
        stats
    )

coverage80_df = pd.DataFrame(
    coverage80_rows
)

coverage80_df.to_csv(
    TABLE_DIR
    / "coverage80_random_subset_tests.csv",
    index=False
)

display(
    coverage80_df
)

# Part F — Freeze system-level selective policies

No task is added to a policy based on Phase 4E curve appearance.

The policy structure was defined before these curves were inspected.

## 24. Define primary and expanded policies

In [ ]:
POLICY_A = {
    "name": (
        "conservative_summary80"
    ),
    "description": (
        "Apply 80% selective coverage only "
        "to PRIMARY-ELIGIBLE summary questions; "
        "answer all factoid, yes/no, and list questions."
    ),
    "task_coverages": {
        "summary": 0.80,
        "factoid": 1.00,
        "yesno": 1.00,
        "list": 1.00,
    },
}

POLICY_B = {
    "name": (
        "expanded_summary_factoid80"
    ),
    "description": (
        "Apply 80% selective coverage to summary "
        "and SECONDARY-ELIGIBLE factoid questions; "
        "answer all yes/no and list questions."
    ),
    "task_coverages": {
        "summary": 0.80,
        "factoid": 0.80,
        "yesno": 1.00,
        "list": 1.00,
    },
}

print(
    json.dumps(
        {
            "Policy A": POLICY_A,
            "Policy B": POLICY_B,
        },
        indent=2
    )
)

## 25. Evaluate a system-level policy

In [ ]:
def evaluate_policy(
    frame,
    policy,
):
    task_rows = []
    selected_parts = []

    for task in TASKS:
        task_df = (
            frame[
                frame[
                    "type"
                ] == task
            ]
            .copy()
        )

        target_coverage = (
            policy[
                "task_coverages"
            ][
                task
            ]
        )

        if (
            target_coverage
            >= 1.0
        ):
            selected = (
                task_df.copy()
            )
        else:
            selected, _ = (
                select_top_fraction(
                    task_df,
                    target_coverage,
                )
            )

        selected[
            "selected_by_policy"
        ] = True

        selected_parts.append(
            selected
        )

        task_rows.append({
            "policy": (
                policy[
                    "name"
                ]
            ),
            "task": task,
            "n_total": int(
                len(task_df)
            ),
            "n_answered": int(
                len(selected)
            ),
            "coverage": float(
                len(selected)
                / len(task_df)
            ),
            "full_task_score": float(
                task_df[
                    "task_score"
                ].mean()
            ),
            "selective_task_score": float(
                selected[
                    "task_score"
                ].mean()
            ),
        })

    per_task = pd.DataFrame(
        task_rows
    )

    selected_all = pd.concat(
        selected_parts,
        ignore_index=True
    )

    full_macro = float(
        per_task[
            "full_task_score"
        ].mean()
    )

    selective_macro = float(
        per_task[
            "selective_task_score"
        ].mean()
    )

    overall_coverage = float(
        len(selected_all)
        / len(frame)
    )

    policy_summary = {
        "policy": (
            policy[
                "name"
            ]
        ),
        "n_total": int(
            len(frame)
        ),
        "n_answered": int(
            len(selected_all)
        ),
        "n_abstained": int(
            len(frame)
            - len(selected_all)
        ),
        "overall_question_coverage": (
            overall_coverage
        ),
        "overall_abstention_rate": float(
            1.0
            - overall_coverage
        ),
        "full_macro_task_score": (
            full_macro
        ),
        "selective_macro_task_score": (
            selective_macro
        ),
        "macro_task_score_gain": float(
            selective_macro
            - full_macro
        ),
    }

    return (
        per_task,
        policy_summary,
        selected_all,
    )

## 26. Evaluate both frozen policies

In [ ]:
policy_task_parts = []
policy_summary_rows = []
policy_selected = {}

for policy in [
    POLICY_A,
    POLICY_B,
]:
    (
        per_task,
        summary,
        selected_all,
    ) = evaluate_policy(
        oof_df,
        policy,
    )

    policy_task_parts.append(
        per_task
    )

    policy_summary_rows.append(
        summary
    )

    policy_selected[
        policy[
            "name"
        ]
    ] = selected_all

policy_task_df = pd.concat(
    policy_task_parts,
    ignore_index=True
)

policy_summary_df = pd.DataFrame(
    policy_summary_rows
)

policy_task_df.to_csv(
    TABLE_DIR
    / "frozen_policy_results_by_task.csv",
    index=False
)

policy_summary_df.to_csv(
    TABLE_DIR
    / "frozen_policy_summary.csv",
    index=False
)

display(
    policy_task_df
)

display(
    policy_summary_df
)

## 27. Policy-level macro gain bootstrap

In [ ]:
def bootstrap_policy_macro_gain(
    frame,
    policy,
    n_boot=BOOTSTRAP_REPS,
    seed=SEED,
):
    # Freeze the original validation selection mask first.
    selected_ids_by_task = {}

    for task in TASKS:
        task_df = (
            frame[
                frame[
                    "type"
                ] == task
            ]
            .copy()
        )

        coverage = (
            policy[
                "task_coverages"
            ][
                task
            ]
        )

        if coverage >= 1:
            selected = task_df
        else:
            selected, _ = (
                select_top_fraction(
                    task_df,
                    coverage,
                )
            )

        selected_ids_by_task[
            task
        ] = set(
            selected[
                "id"
            ].astype(str)
        )

    rng = np.random.default_rng(
        seed
    )

    gains = []

    for _ in range(
        n_boot
    ):
        full_task_scores = []
        selected_task_scores = []

        valid = True

        for task in TASKS:
            task_df = (
                frame[
                    frame[
                        "type"
                    ] == task
                ]
                .reset_index(drop=True)
            )

            n = len(
                task_df
            )

            idx = rng.integers(
                0,
                n,
                size=n
            )

            boot = (
                task_df.iloc[
                    idx
                ]
                .copy()
            )

            full_task_scores.append(
                float(
                    boot[
                        "task_score"
                    ].mean()
                )
            )

            keep = (
                boot[
                    "id"
                ].astype(str)
                .isin(
                    selected_ids_by_task[
                        task
                    ]
                )
            )

            if (
                keep.sum()
                == 0
            ):
                valid = False
                break

            selected_task_scores.append(
                float(
                    boot.loc[
                        keep,
                        "task_score"
                    ].mean()
                )
            )

        if valid:
            gains.append(
                np.mean(
                    selected_task_scores
                )
                - np.mean(
                    full_task_scores
                )
            )

    gains = np.asarray(
        gains,
        dtype=float
    )

    return {
        "policy": (
            policy[
                "name"
            ]
        ),
        "bootstrap_reps_valid": int(
            len(gains)
        ),
        "macro_gain_mean": float(
            gains.mean()
        ),
        "macro_gain_ci95_low": float(
            np.quantile(
                gains,
                0.025
            )
        ),
        "macro_gain_ci95_high": float(
            np.quantile(
                gains,
                0.975
            )
        ),
        "bootstrap_probability_gain_gt_0": float(
            np.mean(
                gains > 0
            )
        ),
    }


policy_bootstrap_df = pd.DataFrame([
    bootstrap_policy_macro_gain(
        oof_df,
        POLICY_A,
        seed=SEED,
    ),
    bootstrap_policy_macro_gain(
        oof_df,
        POLICY_B,
        seed=SEED + 1000,
    ),
])

policy_bootstrap_df.to_csv(
    TABLE_DIR
    / "frozen_policy_macro_gain_bootstrap.csv",
    index=False
)

display(
    policy_bootstrap_df
)

# Part G — Error analysis

Inspect:

- high-confidence failures;
- low-confidence successes.

This helps identify why selective answering succeeds or fails within a task.

## 28. Recover question text when available

In [ ]:
modeling_df = pd.read_csv(
    PHASE4D_MODELING_TABLE,
    dtype={"id": str}
)

analysis_df = (
    oof_df.merge(
        modeling_df[
            [
                "id",
                "type",
                "question_words",
                "ideal_answer_words",
                "mean_top1_entailment",
                "mean_top1_contradiction",
                "top1_supported_claim_rate",
                "top1_contradiction_rate",
            ]
        ],
        on=[
            "id",
            "type",
        ],
        how="left",
        suffixes=(
            "",
            "_model"
        ),
        validate="one_to_one",
    )
)

print(
    "Analysis rows:",
    len(analysis_df)
)

## 29. High-confidence failures

In [ ]:
high_conf_failure_parts = []

for task in TASKS:
    part = (
        analysis_df[
            analysis_df[
                "type"
            ] == task
        ]
        .copy()
    )

    part[
        "failure_priority"
    ] = (
        part[
            RANKING_SCORE
        ]
        - part[
            "task_score"
        ]
    )

    part = (
        part.sort_values(
            [
                "failure_priority",
                RANKING_SCORE,
            ],
            ascending=[
                False,
                False,
            ],
        )
        .head(15)
        .copy()
    )

    high_conf_failure_parts.append(
        part
    )

high_conf_failures = pd.concat(
    high_conf_failure_parts,
    ignore_index=True
)

high_conf_failures.to_csv(
    TABLE_DIR
    / "high_confidence_failures.csv",
    index=False
)

display(
    high_conf_failures[
        [
            "id",
            "type",
            "task_score",
            RANKING_SCORE,
            CALIBRATED_CONFIDENCE,
            "mean_top1_entailment",
            "top1_supported_claim_rate",
            "failure_priority",
        ]
    ]
)

## 30. Low-confidence successes

In [ ]:
low_conf_success_parts = []

for task in TASKS:
    part = (
        analysis_df[
            analysis_df[
                "type"
            ] == task
        ]
        .copy()
    )

    part[
        "success_surprise"
    ] = (
        part[
            "task_score"
        ]
        - part[
            RANKING_SCORE
        ]
    )

    part = (
        part.sort_values(
            [
                "success_surprise",
                "task_score",
            ],
            ascending=[
                False,
                False,
            ],
        )
        .head(15)
        .copy()
    )

    low_conf_success_parts.append(
        part
    )

low_conf_successes = pd.concat(
    low_conf_success_parts,
    ignore_index=True
)

low_conf_successes.to_csv(
    TABLE_DIR
    / "low_confidence_successes.csv",
    index=False
)

display(
    low_conf_successes[
        [
            "id",
            "type",
            "task_score",
            RANKING_SCORE,
            CALIBRATED_CONFIDENCE,
            "mean_top1_entailment",
            "top1_supported_claim_rate",
            "success_surprise",
        ]
    ]
)

# Part H — Freeze Phase 4E

The locked test is still not evaluated.

The frozen Phase 5 protocol will apply:

### Primary analysis
Policy A — conservative summary-only abstention.

### Secondary analysis
Policy B — expanded summary + factoid abstention.

### Negative controls
Yes/no and list remain at full coverage.

For benchmark evaluation, the fixed **coverage policy** is applied by ranking each task with the already-frozen raw Ridge score. No locked-test labels are used to choose which questions are answered.

## 31. Freeze Phase 4E policy

In [ ]:
phase4e_freeze = {
    "dataset": "BioASQ14b",
    "validation_questions": 862,
    "internal_test_evaluated": False,

    "confidence_source": (
        "Phase 4D v2 task-specific OOF confidence"
    ),

    "ranking_signal": (
        RANKING_SCORE
    ),

    "reported_confidence": (
        CALIBRATED_CONFIDENCE
    ),

    "risk_definition": (
        "1 - per-question BioASQ task score"
    ),

    "coverage_selection": {
        "primary_task_coverage": (
            PRIMARY_TASK_COVERAGE
        ),
        "sensitivity_coverages": (
            SENSITIVITY_COVERAGES
        ),
        "best_coverage_selected_post_hoc": False,
    },

    "eligibility_rule": {
        "multiple_testing": (
            "Holm correction over four task-level "
            "raw-Ridge Spearman tests"
        ),
        "primary": (
            "positive Holm-significant Spearman AND "
            "paired-bootstrap MSE improvement over "
            "train-mean baseline"
        ),
        "secondary": (
            "positive Holm-significant Spearman AND "
            "paired-bootstrap MAE improvement, without "
            "MSE improvement, over train-mean baseline"
        ),
        "ineligible": (
            "otherwise"
        ),
    },

    "task_eligibility": (
        eligibility_map
    ),

    "primary_policy": (
        POLICY_A
    ),

    "secondary_policy": (
        POLICY_B
    ),

    "negative_control_tasks": [
        "yesno",
        "list",
    ],

    "locked_test_application": (
        "Rank locked-test questions within each task "
        "using the frozen final task-specific raw Ridge "
        "score. Apply the prespecified coverage fractions. "
        "Do not use locked-test labels for ranking, "
        "coverage selection, or threshold tuning."
    ),

    "policy_a_validation_summary": (
        policy_summary_df[
            policy_summary_df[
                "policy"
            ] == POLICY_A[
                "name"
            ]
        ]
        .iloc[0]
        .to_dict()
    ),

    "policy_b_validation_summary": (
        policy_summary_df[
            policy_summary_df[
                "policy"
            ] == POLICY_B[
                "name"
            ]
        ]
        .iloc[0]
        .to_dict()
    ),

    "internal_test_unlock_allowed": False,

    "next_phase": (
        "Phase 5 — one-time locked internal test evaluation"
    ),
}

with open(
    PHASE4E_DIR
    / "frozen_selective_answering_policy.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        phase4e_freeze,
        f,
        indent=2,
        ensure_ascii=False
    )

print(
    json.dumps(
        phase4e_freeze,
        indent=2,
        ensure_ascii=False
    )
)

## 32. Phase 4E report

In [ ]:
phase4e_report = {
    "dataset": "BioASQ14b",
    "validation_questions": 862,
    "internal_test_evaluated": False,

    "task_eligibility": (
        eligibility_df.to_dict(
            orient="records"
        )
    ),

    "aurc_by_task": (
        aurc_df.to_dict(
            orient="records"
        )
    ),

    "fixed_coverage_results": (
        coverage_table.to_dict(
            orient="records"
        )
    ),

    "coverage80_random_subset_tests": (
        coverage80_df.to_dict(
            orient="records"
        )
    ),

    "policy_results_by_task": (
        policy_task_df.to_dict(
            orient="records"
        )
    ),

    "policy_summary": (
        policy_summary_df.to_dict(
            orient="records"
        )
    ),

    "policy_macro_gain_bootstrap": (
        policy_bootstrap_df.to_dict(
            orient="records"
        )
    ),

    "primary_policy": (
        POLICY_A[
            "name"
        ]
    ),

    "secondary_policy": (
        POLICY_B[
            "name"
        ]
    ),

    "internal_test_evaluated": False,
}

with open(
    PHASE4E_DIR
    / "phase4E_report.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        phase4e_report,
        f,
        indent=2,
        ensure_ascii=False
    )

print(
    json.dumps(
        phase4e_report,
        indent=2,
        ensure_ascii=False
    )
)

# STOP POINT 4E

Do **not** unlock the internal test yet.

Send these outputs for review:

1. **9. Task eligibility**
2. **15. AURC and random-ranking comparisons**
3. **16–19. Risk–coverage figures**
4. **21. Fixed coverage table**
5. **23. 80% coverage random-subset tests**
6. **26. Frozen policy summaries**
7. **27. Policy-level macro gain bootstrap**
8. `frozen_selective_answering_policy.json`
9. `phase4E_report.json`

After review, the entire validation pipeline will be frozen.

Only then should Phase 5 open the 860-question internal test **once**.

## 33. Final safety check

In [ ]:
if not ALLOW_INTERNAL_TEST_EVALUATION:
    print(
        "INTERNAL TEST REMAINS LOCKED."
    )
else:
    raise RuntimeError(
        "Do not unlock the internal test during Phase 4E."
    )

print(
    "Phase 4E output:",
    PHASE4E_DIR
)

print(
    "Primary policy:",
    POLICY_A[
        "name"
    ]
)

print(
    "Secondary policy:",
    POLICY_B[
        "name"
    ]
)

print(
    "Primary task coverage:",
    PRIMARY_TASK_COVERAGE
)

print(
    "Locked-test evaluation:",
    False
)